# Stage 1: Hull Generation (Downsampled)

Generate 2D slice-wise convex hulls from downsampled binary masks.

**Input**: `labels_ds/*.nii.gz`  
**Output**: `labelsHull_ds/*_hull.nii.gz`  
**Metrics**: `metrics/hull_metrics_ds.json`

In [1]:
# Configuration - copy from 00_config.ipynb or run that notebook first
TARGET_DIR = "/mnt/c/users/mwild/firebase/perios/levi_data_1.6.26"

In [2]:
import sys
from pathlib import Path
import numpy as np
from scipy import ndimage
from skimage import morphology

# Add parent directory to path for utils import
sys.path.insert(0, str(Path('.').resolve()))
from utils import (
    load_nifti, save_nifti, save_metrics, ensure_dir,
    largest_component, compute_slicewise_hull, compute_hull_metrics
)

In [3]:
# Setup directories (using downsampled data)
target = Path(TARGET_DIR)
INPUT_DIR = target / "labels_ds"
OUTPUT_DIR = ensure_dir(target / "labelsHull_ds")
METRICS_DIR = ensure_dir(target / "metrics")

print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Metrics: {METRICS_DIR}")

Input:  /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labels_ds
Output: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labelsHull_ds
Metrics: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/metrics


In [4]:
def clean_binary_mask(mask_data):
    """Clean a binary mask by keeping largest component and filling holes."""
    binary_mask = (mask_data > 0).astype(np.uint8)
    binary_mask = largest_component(binary_mask)
    binary_mask = ndimage.binary_fill_holes(binary_mask).astype(np.uint8)
    binary_mask = morphology.binary_closing(binary_mask, footprint=morphology.ball(1))
    return binary_mask.astype(np.uint8)


def process_hull(input_path, output_path):
    """Process a single mask to generate hull."""
    # Load mask
    data, affine, header = load_nifti(input_path)
    
    # Clean the mask
    cleaned_mask = clean_binary_mask(data)
    
    # Compute slice-wise convex hull along axis 2
    hull_mask = compute_slicewise_hull(cleaned_mask, axis=2)
    
    # Post-process
    hull_mask = ndimage.binary_fill_holes(hull_mask).astype(np.uint8)
    hull_mask = largest_component(hull_mask)
    
    # Compute metrics
    metrics = compute_hull_metrics(cleaned_mask, hull_mask)
    
    # Save output
    save_nifti(hull_mask, affine, header, output_path)
    
    return metrics

In [5]:
# Process all masks
mask_files = sorted(INPUT_DIR.glob("*.nii.gz"))
print(f"Found {len(mask_files)} mask files to process")
print("="*60)

all_metrics = []

for idx, mask_file in enumerate(mask_files, 1):
    sample_name = mask_file.stem.replace('.nii', '')
    output_file = OUTPUT_DIR / f"{sample_name}_hull.nii.gz"
    
    print(f"[{idx}/{len(mask_files)}] {mask_file.name}")
    
    try:
        metrics = process_hull(mask_file, output_file)
        metrics['filename'] = mask_file.name
        metrics['sample_name'] = sample_name
        all_metrics.append(metrics)
        
        print(f"    Original: {metrics['original_volume']:,} voxels")
        print(f"    Hull: {metrics['hull_volume']:,} voxels ({metrics['expansion_ratio']:.2f}x)")
        print(f"    Containment: {metrics['containment_ratio']:.4f}")
    except Exception as e:
        print(f"    ERROR: {e}")
        import traceback
        traceback.print_exc()

Found 30 mask files to process
[1/30] Digit10.nii.gz


    Original: 312,691 voxels
    Hull: 479,715 voxels (1.53x)
    Containment: 1.0000
[2/30] Digit105.nii.gz


    Original: 230,064 voxels
    Hull: 308,991 voxels (1.34x)
    Containment: 1.0000
[3/30] Digit12.nii.gz


    Original: 347,395 voxels
    Hull: 531,956 voxels (1.53x)
    Containment: 1.0000
[4/30] Digit2.nii.gz


    Original: 318,876 voxels
    Hull: 508,312 voxels (1.59x)
    Containment: 1.0000
[5/30] Digit23.nii.gz


...

    Original: 217,930 voxels
    Hull: 304,885 voxels (1.40x)
    Containment: 1.0000
[27/30] Digit95.nii.gz


    Original: 316,130 voxels
    Hull: 467,542 voxels (1.48x)
    Containment: 1.0000
[28/30] Digit96.nii.gz


    Original: 191,351 voxels
    Hull: 264,467 voxels (1.38x)
    Containment: 1.0000
[29/30] Digit97.nii.gz


    Original: 280,130 voxels
    Hull: 419,409 voxels (1.50x)
    Containment: 1.0000
[30/30] Digit99.nii.gz


    Original: 221,012 voxels
    Hull: 299,383 voxels (1.35x)
    Containment: 1.0000


In [6]:
# Save metrics
metrics_file = METRICS_DIR / "hull_metrics_ds.json"
save_metrics(all_metrics, metrics_file)

# Summary
print("\n" + "="*60)
print("HULL GENERATION COMPLETE (DOWNSAMPLED)")
print("="*60)
print(f"\nProcessed: {len(all_metrics)}/{len(mask_files)} masks")

if all_metrics:
    avg_expansion = np.mean([m['expansion_ratio'] for m in all_metrics])
    avg_containment = np.mean([m['containment_ratio'] for m in all_metrics])
    print(f"Average expansion ratio: {avg_expansion:.2f}x")
    print(f"Average containment ratio: {avg_containment:.4f}")

print(f"\nOutput: {OUTPUT_DIR}")
print(f"Metrics: {metrics_file}")


HULL GENERATION COMPLETE (DOWNSAMPLED)

Processed: 30/30 masks
Average expansion ratio: 1.47x
Average containment ratio: 1.0000

Output: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labelsHull_ds
Metrics: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/metrics/hull_metrics_ds.json
